# Layer map PoC: paint-order layers for landscape & Impressionist paintings

<a href="https://colab.research.google.com/github/saranunt-0/layer-mapping/blob/main/notebooks/01_poc_layer_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Question this notebook answers:** does fine-tuning an openly licensed depth model on *synthetic, painterly,
layered landscapes* give better layer maps on real paintings than the same model off the shelf?

| | |
|---|---|
| Runtime | GPU. A free **T4** is enough |
| Time | about 45 min end to end (≈5 min data, ≈20 min fine-tune on T4) |
| Model | Depth Anything V2 **Small** (Apache-2.0). Base/Large are CC-BY-NC, so they are not used |
| Training data | Generated procedurally in this notebook, so no third-party rights apply |
| Test paintings | Public-domain works from the Art Institute of Chicago open-access API (CC0) |

No Adobe code, weights or outputs are used anywhere.

**Output convention:** a larger value / higher layer number means *closer to the viewer* (painted later).

## 0. Setup

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/saranunt-0/layer-mapping.git"
BRANCH = "main"
REPO_DIR = "/content/layer-mapping"

if not os.path.exists(REPO_DIR):
    url = REPO_URL
    try:
        # Only needed while the repo is private: add a GITHUB_TOKEN secret (key icon, left sidebar).
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
sys.path.insert(0, REPO_DIR)

import torch, transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.45"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"], check=True)
    raise SystemExit("transformers was upgraded: Runtime > Restart session, then run all cells again.")
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - Runtime > Change runtime type > GPU")
print("CPU cores:", os.cpu_count())

In [ ]:
import json, time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from layermap import data, layers, metrics, synth
from layermap import model as lm
from layermap import train as lt


def load_rgb(path, max_side=1024):
    rgb = cv2.imread(str(path), cv2.IMREAD_COLOR)[..., ::-1]
    s = max_side / max(rgb.shape[:2])
    if s < 1:
        rgb = cv2.resize(rgb, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(rgb)


def show_row(images, titles, size=4.0):
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for ax, im, t in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(im)
        ax.set_title(t, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def mean_metrics(rows):
    return {k: round(float(np.nanmean([r[k] for r in rows])), 4) for k in rows[0]}


def prior_metrics(ds):
    """Score of the "lower in the frame = closer" rule on an eval dataset (per target if several)."""
    names = getattr(ds, "targets", ("layer",))
    rows = []
    for i in range(len(ds)):
        t = ds[i][1]
        t = t[None] if t.dim() == 2 else t
        row = {}
        for c, n in enumerate(names):
            pre = "" if len(names) == 1 else f"{n}/"
            row.update({pre + k: v for k, v in metrics.all_metrics(metrics.vertical_prior(t[c].shape), t[c]).items()})
        rows.append(row)
    return mean_metrics(rows)

## 1. Test paintings (public domain)

In [ ]:
import requests

PAINT_DIR = Path("/content/paintings")
PAINT_DIR.mkdir(exist_ok=True)
QUERIES = ["Monet landscape", "Pissarro landscape", "Sisley landscape", "Caillebotte", "Renoir landscape",
           "Cezanne landscape"]
PER_QUERY = 2
HEADERS = {"AIC-User-Agent": "layer-mapping PoC (non-commercial research)"}


def fetch_aic(query, n):
    """Public-domain paintings from the Art Institute of Chicago open-access API (images are CC0)."""
    r = requests.get("https://api.artic.edu/api/v1/artworks/search", headers=HEADERS, timeout=30, params={
        "q": query, "limit": 25, "query[term][is_public_domain]": "true",
        "fields": "id,title,artist_title,image_id,is_public_domain,artwork_type_title"})
    r.raise_for_status()
    got = []
    for a in r.json().get("data", []):
        if not (a.get("is_public_domain") and a.get("image_id")):
            continue
        if (a.get("artwork_type_title") or "Painting") != "Painting":
            continue
        out = PAINT_DIR / f"aic_{a['id']}.jpg"
        if not out.exists():
            img = requests.get(f"https://www.artic.edu/iiif/2/{a['image_id']}/full/843,/0/default.jpg",
                               headers=HEADERS, timeout=60)
            img.raise_for_status()
            out.write_bytes(img.content)
        got.append(out)
        if len(got) == n:
            break
    return got


for q in QUERIES:
    try:
        print(q, "->", [p.name for p in fetch_aic(q, PER_QUERY)])
    except Exception as e:
        print(q, "-> failed:", e)

In [ ]:
# Optional: add your own paintings. Set UPLOAD = True and pick files when the button appears.
UPLOAD = False
if UPLOAD:
    from google.colab import files
    for name, content in files.upload().items():
        (PAINT_DIR / name).write_bytes(content)

paintings = sorted(p for p in PAINT_DIR.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"))
print(len(paintings), "paintings")
if not paintings:
    print("No paintings: the museum download failed and nothing was uploaded. Set UPLOAD = True and re-run.")

## 2. Zero-shot baseline

Depth Anything V2 Small, unchanged, predicts relative inverse depth. Landscape painters work roughly back to front
(sky → distance → middle ground → foreground), so physical depth is already a reasonable stand-in for paint order.
**This is the bar the fine-tuned model has to beat.**

The continuous map is turned into `N_LAYERS` discrete layers with superpixels plus 1-D k-means
(`layermap.layers.depth_to_layers`), so layer edges follow colour edges in the painting.

In [ ]:
N_LAYERS = 5
INFER_SIZE = 518  # long side fed to the network; 770 or 1036 gives finer edges at more cost

base = lm.load_model(lm.DEFAULT_BACKBONE)
zero_shot = {}
for p in paintings:
    rgb = load_rgb(p)
    d = lm.predict(base, rgb, size=INFER_SIZE)
    zero_shot[p] = (rgb, d, layers.depth_to_layers(d, N_LAYERS, rgb=rgb))
    show_row([rgb, layers.colorize_depth(d), layers.colorize(zero_shot[p][2], N_LAYERS)],
             [p.name, "zero-shot depth (bright = near)", f"{N_LAYERS} layers (bright = front)"])

## 3. Synthetic training data

`layermap.synth` paints each scene back to front (sky, clouds, far ranges, water, fields, trees/houses,
foreground, flowers) with thousands of jittered brush strokes. Every stroke also writes its layer number into an
ID buffer, so the label is exact by construction. About 15% of scenes are flat fills (vector-illustration look).

In [ ]:
DATA = Path("/content/data")
N_TRAIN, N_VAL = 2000, 200
cfg_synth = synth.SynthConfig(size=576)

t = time.time()
data.generate_dataset(DATA / "train", N_TRAIN, start_seed=0, cfg=cfg_synth)
data.generate_dataset(DATA / "val", N_VAL, start_seed=1_000_000, cfg=cfg_synth)
print(f"generated in {time.time() - t:.0f}s on {os.cpu_count()} cores")

samples = data.list_pairs(DATA / "val")[:4]
show_row([load_rgb(i) for i, _ in samples] + [layers.colorize(data.read_label(l)) for _, l in samples],
         ["synthetic"] * 4 + ["label"] * 4, size=3)

## 4. Metrics, and why plain accuracy is not enough

All metrics sample pixel pairs from different layers and check that the predicted order is right:

* `acc_all`: uniform random pairs
* `acc_local`: pairs a few pixels apart, i.e. across layer boundaries
* `acc_hard`: pairs where the **upper** pixel is in front (a tree against the hill behind it, for example)

The rule "lower in the frame = closer" already scores about 0.9 on `acc_all` for landscapes, and by definition
0 on `acc_hard`. **Judge models on `acc_local` and `acc_hard`.**

In [ ]:
val_ds = data.LayerFolderDataset(DATA / "val", 518, train=False)
results = {"vertical prior": prior_metrics(val_ds), "zero-shot": lt.evaluate(base, val_ds)}
for k, v in results.items():
    print(f"{k:15s}", {m: round(x, 4) for m, x in v.items()})

## 5. Fine-tune

1,500 steps × batch 8 at 518 px. On a T4 this is about 15-25 min (the log prints `s/step`).
Checkpoints go to `/content/runs/poc/{last,best}`.

In [ ]:
cfg_train = lt.TrainConfig(
    train_dirs=[str(DATA / "train")], val_dir=str(DATA / "val"), out_dir="/content/runs/poc",
    steps=1500, batch_size=8, warmup=100, eval_every=500, save_every=500, num_workers=min(4, os.cpu_count()),
)
history = lt.train(cfg_train)

train_log = [h for h in history if "loss" in h]
plt.figure(figsize=(6, 3))
plt.plot([h["step"] for h in train_log], [h["loss"] for h in train_log])
plt.xlabel("step"); plt.ylabel("train loss"); plt.show()

## 6. Compare: synthetic validation set and real paintings

In [ ]:
ft = lm.load_model("/content/runs/poc/best")
results["fine-tuned"] = lt.evaluate(ft, val_ds)
for k, v in results.items():
    print(f"{k:15s}", {m: round(x, 4) for m, x in v.items()})

In [ ]:
for p in paintings:
    rgb, d0, lab0 = zero_shot[p]
    d1 = lm.predict(ft, rgb, size=INFER_SIZE)
    lab1 = layers.depth_to_layers(d1, N_LAYERS, rgb=rgb)
    show_row([rgb, layers.colorize(lab0, N_LAYERS), layers.colorize(lab1, N_LAYERS), layers.colorize_depth(d1)],
             [p.name, "zero-shot layers", "fine-tuned layers", "fine-tuned layer depth"], size=3.5)

## 7. Score on hand-labelled paintings (strongly recommended)

Synthetic scores only show the model learned *the generator*. Before spending GPU hours, label 15-30 real paintings:

1. Open a painting in Krita / GIMP / Photopea and paint a **grey-level layer map** on a new layer:
   black = furthest back, brighter = closer, same grey = same layer. 3-7 layers is plenty.
   Use a hard-edged brush with anti-aliasing off; blended edge pixels would count as extra layers.
2. Export it as PNG with the **same file name** as the painting.
3. Zip as `real_eval/images/*.jpg` + `real_eval/labels/*.png` and upload, or put it on Drive.

Never train on these; they are your test set.

In [ ]:
REAL_DIR = Path("/content/real_eval")  # or a Drive path after mounting
# To upload a zip:  from google.colab import files; z = files.upload(); !unzip -q -o {list(z)[0]} -d /content
if (REAL_DIR / "labels").exists():
    real_ds = data.LayerFolderDataset(REAL_DIR, 518, train=False)
    real = {"vertical prior": prior_metrics(real_ds), "zero-shot": lt.evaluate(base, real_ds),
            "fine-tuned": lt.evaluate(ft, real_ds)}
    for k, v in real.items():
        print(f"{k:15s}", {m: round(x, 4) for m, x in v.items()})
else:
    print("No labelled paintings yet: see the instructions above.")

## 8. Export layers for one painting

In [ ]:
import shutil

PICK = paintings[0]          # change to any path in `paintings`
N_EXPORT = 6
rgb = load_rgb(PICK, max_side=2048)
labels = layers.depth_to_layers(lm.predict(ft, rgb, size=INFER_SIZE), N_EXPORT, rgb=rgb)
out_dir = Path("/content/export") / PICK.stem
layers.export_layers(rgb, labels, out_dir)
zip_path = shutil.make_archive(str(out_dir), "zip", out_dir)
print("wrote", zip_path)
# from google.colab import files; files.download(zip_path)

## Decision gate

Go on to `02_train_full.ipynb` only if:

1. On **your hand-labelled paintings** (step 7), fine-tuned beats zero-shot on `acc_local` **and** `acc_hard`
   by a clear margin (with ~20 paintings, treat differences under ~2 points as noise), **and**
2. the fine-tuned layer maps look cleaner by eye on most paintings in step 6.

Other outcomes and what they mean:

* **Wins on synthetic, loses on real paintings.** The generator does not look enough like real paintings
  (domain gap). Improve `layermap/synth.py` (more stroke styles, colour variety, compositions) before scaling up.
* **Zero-shot already wins on real paintings.** Use the zero-shot + superpixel pipeline as is. Training is not
  worth the GPU hours yet.
* **Both are poor on `acc_hard`.** The model cannot separate foreground objects from what is behind them.
  Add hand-labelled training paintings (a few hundred, kept separate from the eval set) via
  `EXTRA_TRAIN_DIRS` in notebook 02.